# Demonstracao Tecnica — Pipeline SIH/SUS (AV1)

**Pergunta de pesquisa:**
> Como as ondas do COVID-19 impactaram o volume de internacoes e a mortalidade hospitalar no SUS-SP entre 2020 e 2023?

Este notebook demonstra as etapas implementadas para a **AV1**:
1. **Ingestao** — leitura de arquivo `.dbc` individual (camada bronze)
2. **Armazenamento** — schema, tipos e qualidade dos dados
3. **Transformacao** — aplicacao das funcoes `src/transform.py` sobre a camada silver

> A analise completa (series temporais, perfil demografico, storytelling) sera realizada na **AV2**.

---
**Fonte:** SIH/SUS — DATASUS, AIH Reduzida  
**Recorte:** Estado de Sao Paulo, Jan/2020 a Dez/2023  
**Arquivos bronze:** 48 `.dbc`  
**Silver gerada:** 9.635.546 registros

## 0. Configuracao

In [ ]:
import sys
import os
from pathlib import Path

ROOT = Path(os.getcwd())
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd

from src.ingest import read_dbc, clean_dtypes, list_dbc_files
from src.transform import add_covid_flag, add_time_columns, add_age_group, COVID_CIDS

BRONZE_DIR  = ROOT / 'data' / 'bronze' / 'sihsus' / 'SP'
SILVER_FILE = ROOT / 'data' / 'silver' / 'sihsus_sp.parquet'

print(f'Root: {ROOT}')
print(f'Silver disponivel: {SILVER_FILE.exists()}')

## 1. Ingestao — Leitura de arquivo .dbc (Bronze)

In [ ]:
# Lista todos os arquivos disponiveis na camada bronze
files = list_dbc_files(BRONZE_DIR)
print(f'Total de arquivos .dbc: {len(files)}')
print(f'Primeiro: {files[0].name}  |  Ultimo: {files[-1].name}')

In [ ]:
# Demonstracao: le e processa Abril/2021 (pico da segunda onda — variante Gama)
print('Lendo RDSP2104.dbc...')
df_sample = read_dbc(BRONZE_DIR / 'RDSP2104.dbc')
df_sample = clean_dtypes(df_sample)

print(f'Registros: {len(df_sample):,}')
print(f'Campos selecionados: {len(df_sample.columns)}')
df_sample.head(3)

In [ ]:
# Schema e tipos gerados pelo clean_dtypes
df_sample.dtypes

## 2. Armazenamento — Qualidade dos dados

In [ ]:
# Verificacao de nulos
nulls = df_sample.isnull().sum()
nulls_pct = (nulls / len(df_sample) * 100).round(2)
resultado = pd.DataFrame({'nulos': nulls, 'pct': nulls_pct}).query('nulos > 0')
print(resultado if not resultado.empty else 'Nenhum campo nulo neste arquivo.')

In [ ]:
# Estatisticas descritivas dos campos numericos principais
df_sample[['DIAS_PERM', 'UTI_MES_TO', 'IDADE', 'VAL_TOT', 'MORTE']].describe().round(2)

## 3. Transformacao — Aplicacao das funcoes src/transform.py

In [ ]:
# Nota sobre o CID do COVID-19 no SIH/SUS
print(f'CID usado pelo DATASUS para COVID-19: {COVID_CIDS}')
print()
print('O SIH/SUS usa B342 (Infeccao por coronavirus NE).')
print('Os codigos U07.1/U07.2 do CID-10 oficial nao aparecem nos arquivos 2020-2023.')
print()

# Demonstracao da flag COVID no arquivo de abril/2021
df_sample = add_covid_flag(df_sample)
n_covid = df_sample['is_covid'].sum()
n_total = len(df_sample)
print(f'Abril/2021 — {n_covid:,} internacoes COVID de {n_total:,} totais ({100*n_covid/n_total:.1f}%)')

In [ ]:
# Demonstracao das colunas derivadas de tempo
df_sample = add_time_columns(df_sample)
print('Colunas de tempo adicionadas:')
print(df_sample[['DT_INTER', 'ano', 'mes', 'ano_mes']].head(5).to_string())

In [ ]:
# Demonstracao da faixa etaria
df_sample = add_age_group(df_sample)
print('Distribuicao por faixa etaria (Abril/2021):')
print(df_sample['faixa_etaria'].value_counts().sort_index())

## 4. Silver — Resumo da camada consolidada

A camada silver contem todos os 48 arquivos processados e consolidados.
Este resumo confirma que o pipeline de ingestao foi executado com sucesso.

In [ ]:
df_silver = pd.read_parquet(SILVER_FILE)

print('=== Resumo da camada silver ===')
print(f'Total de registros:  {len(df_silver):>10,}')
print(f'Periodo:             {df_silver["DT_INTER"].min().date()} a {df_silver["DT_INTER"].max().date()}')
print(f'Colunas:             {list(df_silver.columns)}')
print(f'Tamanho em memoria:  {df_silver.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print()
print('=== Registros por ano de internacao ===')
print(df_silver['DT_INTER'].dt.year.value_counts().sort_index())

In [ ]:
# Aplicando transformacoes no silver completo para validacao
df_silver = df_silver[df_silver['DT_INTER'].dt.year.between(2020, 2023)].copy()
df_silver = add_covid_flag(df_silver)

print('=== Validacao das transformacoes (2020-2023) ===')
print(f'Total de internacoes:   {len(df_silver):>10,}')
print(f'Internacoes COVID (B342):{df_silver["is_covid"].sum():>9,}  ({100*df_silver["is_covid"].mean():.1f}%)')
print(f'Obitos totais:          {int(df_silver["MORTE"].sum()):>10,}  ({100*df_silver["MORTE"].mean():.1f}%)')